## Spotify Tracks Modeling (Model A scores)

### What this notebook does

Loads a **tuned estimator** saved by `01d_spotify_model_baseline.ipynb` from `notebooks/models/`, scores all rows in `spotify_tracks_baseline.parquet`, and writes **`data/processed/preds_model_a.parquet`** (`p_a`, `split`, `viral`) for `07_meta_combiner.ipynb`.

### How to read `01d` (two different “F1” stories)

**1. Hyperparameter search (`RandomizedSearchCV` / `GridSearchCV`)** prints **`Best CV F1-Score`** = `best_score_`: mean **cross-validated** F1. In `01d`, `scoring="f1"` uses the **positive class (viral = 1)** by default.

| Saved model | Best CV F1 (`best_score_` in notebook output) |
|-------------|-----------------------------------------------|
| `rf_tuned_randomsearch.joblib` | 0.503 |
| `rf_tuned_gridsearch.joblib` | 0.509 |
| `xgb_tuned_gridsearch.joblib` | 0.648 |
| **`xgb_tuned_randomized.joblib`** | **0.652** (highest CV) |

**2. Hold-out test set** — cells also print **`precision_score` / `recall_score` / `f1_score`** on `y_test`: these are **only for viral = 1** (sklearn binary default). **`classification_report`** adds **per-class** rows: label **`0`** = non-viral, **`1`** = viral.

Approximate **test** metrics from **stored outputs** in `01d`:

| Model | Test accuracy | `f1_score` (viral=1) | Class 0 F1 (non-viral, from report) |
|-------|----------------|----------------------|-------------------------------------|
| `rf_tuned_randomsearch.joblib` | 0.817 | 0.52 | 0.89 |
| `rf_tuned_gridsearch.joblib` | 0.818 | **0.53** | 0.89 |
| `xgb_tuned_randomized.joblib` | 0.802 | 0.39 | 0.88 |
| `xgb_tuned_gridsearch.joblib` | 0.802 | 0.39 | 0.88 |

Your **Accuracy ~0.82** and **Precision / Recall ~0.94 / 0.84** match the **non-viral** line and the **“Precision/Recall of non-viral”** prints — not the single-line **`F1 Score`** from `f1_score(...)`, which is **viral-only** and stays near **0.39–0.53** for these models.

**Default here:** **`rf_tuned_gridsearch.joblib`** — best **test F1 on viral (1)** among the four saved models. For **best CV score** during tuning, use **`xgb_tuned_randomized.joblib`** instead. Override with **`MODEL_A_FILE`** (or env) in the paths cell.

### Inputs
- `data/processed/spotify_tracks_baseline.parquet` (same feature columns as in `01d`)
- `data/processed/spotify_tracks_viral.parquet` (for `track_id` and row alignment)
- `notebooks/models/<MODEL_A_FILE>`

### Output columns
`row_id`, `track_id`, `viral`, `split`, `p_a`


### 1. Path & sanity checks

In [4]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

BASELINE = Path("../data/processed/spotify_tracks_baseline.parquet")
SPOTIFY_VIRAL = Path("../data/processed/spotify_tracks_viral.parquet")
MODELS_DIR = Path("../notebooks/models")
# Default: best test F1 on viral (1) among 01d saved models; use xgb_tuned_randomized.joblib for best CV F1
MODEL_A_FILE = os.environ.get("MODEL_A_FILE", "xgb_tuned_randomized.joblib")
MODEL_A_PATH = MODELS_DIR / MODEL_A_FILE
OUT_A = Path("../data/processed/preds_model_a.parquet")

print("BASELINE:", BASELINE.exists(), BASELINE)
print("SPOTIFY_VIRAL:", SPOTIFY_VIRAL.exists(), SPOTIFY_VIRAL)
print("MODEL_A_PATH:", MODEL_A_PATH.exists(), MODEL_A_PATH)

BASELINE: True ../data/processed/spotify_tracks_baseline.parquet
SPOTIFY_VIRAL: True ../data/processed/spotify_tracks_viral.parquet
MODEL_A_PATH: True ../notebooks/models/xgb_tuned_randomized.joblib


In [5]:
A = pd.read_parquet(BASELINE)
sp = pd.read_parquet(SPOTIFY_VIRAL)

print("A", A.shape, "sp", sp.shape)
assert len(A) == len(sp)
assert (A["viral"].astype(int).values == sp["viral"].astype(int).values).mean() > 0.999

row_id = np.arange(len(A), dtype=np.int64)


A (113999, 128) sp (113999, 22)


In [6]:
keys = sp[["track_id", "viral"]].drop_duplicates("track_id")
train_ids, temp_ids = train_test_split(
    keys["track_id"],
    test_size=0.3,
    random_state=42,
    stratify=keys["viral"],
)
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.5,
    random_state=42,
    stratify=keys.set_index("track_id").loc[temp_ids, "viral"],
)

id_to_split = {}
for tid in train_ids:
    id_to_split[tid] = "train"
for tid in val_ids:
    id_to_split[tid] = "val"
for tid in test_ids:
    id_to_split[tid] = "test"

split = sp["track_id"].map(id_to_split)
split.value_counts()


track_id
train    79732
test     17145
val      17122
Name: count, dtype: int64

In [7]:
X = A.drop(columns=["viral"]).copy()
y = A["viral"].astype(int).values

train_mask = (split == "train").values
val_mask = (split == "val").values
test_mask = (split == "test").values

if not MODEL_A_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MODEL_A_PATH}. Train/tune in 01d and save joblib under notebooks/models/."
    )

model_a = joblib.load(MODEL_A_PATH)
p_a = model_a.predict_proba(X)[:, 1]

for name, mask in [("val", val_mask), ("test", test_mask)]:
    print(
        name,
        "ROC-AUC=",
        roc_auc_score(y[mask], p_a[mask]),
        "PR-AUC=",
        average_precision_score(y[mask], p_a[mask]),
    )

preds_a = pd.DataFrame(
    {
        "row_id": row_id,
        "track_id": sp["track_id"].values,
        "viral": A["viral"].astype(int).values,
        "split": split.values,
        "p_a": p_a,
    }
)
OUT_A.parent.mkdir(parents=True, exist_ok=True)
preds_a.to_parquet(OUT_A, index=False)
OUT_A


val ROC-AUC= 0.8553020064581662 PR-AUC= 0.6804935648646604
test ROC-AUC= 0.8512767710704817 PR-AUC= 0.6748689715678249


PosixPath('../data/processed/preds_model_a.parquet')